<a href="https://colab.research.google.com/github/iav2002/AppliedDeepLearning/blob/main/Part3_3_TransferLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Part 3.3 - Backbone Fine Tune

ResNet50 on Block 1 for age category classification. Two runs, frozen backbone with head only, then full fine tune.

Compared against notebook 8 (BestModel from scratch) and notebook 10 (autoencoder transfer).

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Colab Notebooks/AppliedDL/face_age.zip" /content/
!cp -r "/content/drive/MyDrive/Colab Notebooks/AppliedDL/data_splits" /content/
!unzip -q /content/face_age.zip -d /content/

## 2. Imports

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet50_Weights
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.pyplot as plt
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

cuda
NVIDIA A100-SXM4-40GB


## 3. Split Block 1 for classification

Block 1 split 80/20 train/val, stratified by age_category to keep class balance. Same approach used on Block 2 in notebook 10.

In [4]:
b1 = pd.read_csv("/content/data_splits/block1.csv")
b1_train, b1_val = train_test_split(
    b1, test_size=0.2, stratify=b1["age_category"], random_state=42
)
b1_train.to_csv("/content/b1_train.csv", index=False)
b1_val.to_csv("/content/b1_val.csv", index=False)
print(f"block1 total {len(b1)}  train {len(b1_train)}  val {len(b1_val)}")
print(b1["age_category"].value_counts())

block1 total 4392  train 3513  val 879
age_category
infant    959
youth     732
mature    616
mid       587
senior    583
child     506
teen      409
Name: count, dtype: int64


## 4. Data

In [6]:
CATEGORIES = ["infant", "child", "teen", "youth", "mid", "mature", "senior"]
CAT_TO_IDX = {c: i for i, c in enumerate(CATEGORIES)}


class FaceAgeDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"/content/{row['path']}").convert("RGB")
        if self.transform:
            img = self.transform(img)
        lbl = torch.tensor(CAT_TO_IDX[row["age_category"]], dtype=torch.long)
        return img, lbl

## 5. Transforms

Resize to 224x224 because resnet50 was pretrained at that size. Same augmentation preset as the rest of the project on train, clean transform on val

In [7]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

tf_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

tf_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

In [8]:
train_ds = FaceAgeDataset("/content/b1_train.csv", transform=tf_train)
val_ds = FaceAgeDataset("/content/b1_val.csv", transform=tf_val)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

print(f"train batches {len(train_loader)}  val batches {len(val_loader)}")
imgs, lbls = next(iter(train_loader))
print(f"batch shape {tuple(imgs.shape)}  labels shape {tuple(lbls.shape)}")

train batches 55  val batches 14
batch shape (64, 3, 224, 224)  labels shape (64,)


## 6. Train and eval functions

In [12]:
def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None):
    model.train()
    total = 0
    n = 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = loss_fn(preds, lbls)
        loss.backward()
        optimizer.step()
        total += loss.item() * imgs.size(0)
        n += imgs.size(0)
    if scheduler is not None:
        scheduler.step()
    return total / n


def evaluate(model, loader, loss_fn):
    model.eval()
    total = 0
    n = 0
    correct = 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds = model(imgs)
            loss = loss_fn(preds, lbls)
            total += loss.item() * imgs.size(0)
            n += imgs.size(0)
            correct += (preds.argmax(1) == lbls).sum().item()
    return total / n, correct / n


def run_variant(model, train_loader, val_loader, max_epochs=30, patience=8, lr=1e-3):
    # train only the params that need grad, lets the frozen run reuse this helper cleanly
    params = [p for p in model.parameters() if p.requires_grad]
    print(f"trainable params {sum(p.numel() for p in params):,}", flush=True)

    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs, eta_min=1e-5)

    best_acc = -1.0
    best_state = None
    hist = {"train": [], "val_loss": [], "val_acc": []}
    no_improve = 0

    t0 = time.time()
    for epoch in range(max_epochs):
        tr = train_one_epoch(model, train_loader, loss_fn, opt, sched)
        vl, va = evaluate(model, val_loader, loss_fn)
        hist["train"].append(tr)
        hist["val_loss"].append(vl)
        hist["val_acc"].append(va)
        print(f"epoch {epoch+1:2d}  train {tr:.4f}  val {vl:.4f}  acc {va:.4f}", flush=True)

        if va > best_acc:
            best_acc = va
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"early stop at epoch {epoch+1}", flush=True)
                break

    print(f"best acc {best_acc:.4f}  time {time.time() - t0:.1f}s", flush=True)
    return {"best_acc": best_acc, "best_state": best_state, "hist": hist}

## 7. Model builder

Loads resnet50, optionally freezes the backbone, swaps the classifier for our 7-way head. Bestmodel mirrors the head used in notebook 8.

In [14]:
def build_resnet50(head_type, freeze_backbone, n_classes=7):
    weights = ResNet50_Weights.IMAGENET1K_V2
    m = models.resnet50(weights=weights)

    if freeze_backbone:
        for p in m.parameters():
            p.requires_grad = False

    in_feats = m.fc.in_features  # 2048
    if head_type == "simple":
        m.fc = nn.Linear(in_feats, n_classes)
    else:
        m.fc = nn.Sequential(
            nn.Linear(in_feats, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, n_classes),
        )
    return m


# sanity check before running anything heavy
m = build_resnet50(head_type="simple", freeze_backbone=True).to(device)
n_total = sum(p.numel() for p in m.parameters())
n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"frozen simple, total {n_total:,}  trainable {n_train:,}")

x = torch.randn(2, 3, 224, 224).to(device)
print(f"output {tuple(m(x).shape)}")

frozen simple, total 23,522,375  trainable 14,343
output (2, 7)
